In [ ]:
# --- Imports ---

import copy
import hashlib
import json
import math
import os
import random
import warnings
from collections import defaultdict
from collections.abc import Sized
from pathlib import Path
from typing import Dict, List, Optional, Tuple, TypedDict

import h5py
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from sklearn.metrics import ConfusionMatrixDisplay, average_precision_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

# Visualization libraries
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# Model inspection tools
import torchinfo

# Interpretability tools
from captum.attr import IntegratedGradients
from captum.attr import visualization as viz


In [ ]:
# --- Check running mode and load dataset ---

# Define project paths for Colab and local environments
COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks/CHE1148/project_code/CHE1148_Defect_Detecting")
LOCAL_PROJECT_ROOT = Path("C:\Download\Pycharm code\CHE1148_Defect_Detecting")

# Detect if running in Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

# Set project root based on environment
if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = COLAB_PROJECT_ROOT
    runtime_mode = "colab_drive"
else:
    PROJECT_ROOT = LOCAL_PROJECT_ROOT
    runtime_mode = "local"

# Validate project root and set working directory
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
print(f"Runtime mode: {runtime_mode}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# --- Paths and Device Setup ---

# Use PROJECT_ROOT from previous cell
DATA_ROOT = PROJECT_ROOT / "data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Data directory structure
RAW = DATA_ROOT / "raw" / "textile"
PROCESSED = DATA_ROOT / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = PROCESSED / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# File path definitions
TRAIN_H5, TRAIN_CSV = RAW / "train64.h5", RAW / "train64.csv"
TEST_H5, TEST_CSV = RAW / "test64.h5", RAW / "test64.csv"
OUT_H5, OUT_CSV = PROCESSED / "full64.h5", PROCESSED / "full64.csv"
TRAIN_SPLIT_CSV = PROCESSED / "train_split.csv"
VAL_SPLIT_CSV = PROCESSED / "val_split.csv"
TEST_SPLIT_CSV = PROCESSED / "test_split.csv"

# Device setup
def select_device(require_cuda: bool = False):
    if torch.cuda.is_available() and torch.cuda.device_count() > 0:
        return torch.device("cuda"), f"cuda ({torch.cuda.get_device_name(0)})"
    if require_cuda: raise RuntimeError("CUDA unavailable")
    return torch.device("cpu"), "cpu"

REQUIRE_CUDA = False
device, device_name = select_device(require_cuda=REQUIRE_CUDA)
print(f"Using device: {device_name}")

In [ ]:
# --- Utility Functions ---

# Check if a file exists, raise error if missing
def _require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

# Convert label to string and remove leading/trailing whitespace
def _normalize_label(x) -> str:
    return str(x).strip()

# Print the number of samples per class in a DataFrame
def print_class_counts(df: pd.DataFrame, title: str) -> None:
    if "indication_type" not in df.columns:
        return
    vc = df["indication_type"].astype(str).str.strip().value_counts()
    print(f"\n[{title}] total_images={len(df)}")
    for k, v in vc.items():
        print(f"  {k}: {v}")


In [ ]:
# --- Global Training and Evaluation Config ---

# Reproducibility
GLOBAL_SEED = 42

def set_seed(seed: int = GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(GLOBAL_SEED)

FULL_CLASSES = ["good", "color", "cut", "hole", "thread", "metal_contamination"]

DEFAULT_NUM_WORKERS = 2 if bool(globals().get("IN_COLAB", False)) else 0

# --- Global Optimizer Configuration ---
OPTIM_CFG = {
    "name": "adam",
    "lr": float(0.001),
    "foreach": bool(False),
}

# --- Global Evaluation Configuration ---
EVAL_CFG = {
    "f1_average": "macro",
    "auprc_average": "macro",
    "zero_division": int(0),
    "early_stop_metric": "f1",
    "early_stop_mode": "max",
}

# --- Global Training Configuration ---
TRAIN_CFG = {
    "batch": int(128),
    "epochs": int(20),
    "patience": int(5),
    "num_workers": int(DEFAULT_NUM_WORKERS),
}

In [ ]:
# --- Training Utilities ---

# Monitors a metric and stops training if it stops improving
class EarlyStopping:
    def __init__(self, patience=5, verbose=True, mode="min", metric_name="metric", min_delta=0.0):
        if mode not in {"min", "max"}: raise ValueError("mode must be 'min' or 'max'")
        self.patience, self.verbose, self.mode = patience, verbose, mode
        self.metric_name, self.min_delta = metric_name, min_delta
        self.counter, self.early_stop = 0, False
        self.best_score = float("inf") if mode == "min" else -float("inf")
        self.best_model_state = None

    # Check if the current score improves upon the best recorded score
    def _is_improvement(self, score):
        if self.mode == "min": return score < (self.best_score - self.min_delta)
        return score > (self.best_score + self.min_delta)

    # Update early stopping status based on the latest validation score
    def __call__(self, score, model):
        if self._is_improvement(score):
            self.best_score, self.counter = score, 0
            self.best_model_state = copy.deepcopy(model.state_dict())
            if self.verbose: print(f"Validation {self.metric_name} improved. Saving weights.")
        else:
            self.counter += 1
            if self.verbose: print(f"EarlyStopping counter: {self.counter} of {self.patience}")
            if self.counter >= self.patience: self.early_stop = True

# Compute macro AUPRC only for classes present in y_true
def _compute_macro_auprc_safe(y_true, y_prob, num_classes):
    y_true_bin = np.eye(num_classes)[y_true]
    ap_values = []

    for cls_id in range(num_classes):
        y_cls = y_true_bin[:, cls_id]
        if not np.any(y_cls == 1):
            continue
        try:
            with warnings.catch_warnings():  # type: ignore[var-annotated]
                warnings.filterwarnings(
                    "ignore",
                    message="No positive class found in y_true, recall is set to one for all thresholds.",
                    category=UserWarning,
                )
                ap = average_precision_score(y_cls, y_prob[:, cls_id])
            ap_values.append(float(ap))
        except Exception:
            continue

    if not ap_values:
        return float("nan")
    return float(np.mean(ap_values))

# Calculate Accuracy, F1-score, and AUPRC
def _compute_eval_metrics(y_true, y_pred, y_prob, num_classes):
    acc = 100.0 * (y_pred == y_true).sum() / max(len(y_true), 1)
    f1 = f1_score(y_true, y_pred, labels=list(range(num_classes)),
                  average=EVAL_CFG["f1_average"], zero_division=int(EVAL_CFG["zero_division"]))
    auprc = _compute_macro_auprc_safe(y_true, y_prob, num_classes)
    return {"accuracy": float(acc), "f1": float(f1), "auprc": float(auprc)}

# Convert a numeric metric value to a formatted string, handling NaNs
def _metric_to_str(v, digits=4):
    return "nan" if np.isnan(v) else f"{v:.{digits}f}"

# Execute a single training or validation epoch across the provided loader
def run_step(model, loader, criterion, optimizer, device, is_train=True):
    model.train() if is_train else model.eval()
    total_loss, all_true, all_pred, all_prob = 0.0, [], [], []

    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            all_true.append(labels.cpu().numpy())
            all_pred.append(outputs.argmax(dim=1).cpu().numpy())
            all_prob.append(torch.softmax(outputs, dim=1).detach().cpu().numpy())

    avg_loss = total_loss / max(len(loader), 1)
    if not all_true: return avg_loss, {"accuracy": 0.0, "f1": 0.0, "auprc": float("nan")}

    y_true, y_pred, y_prob = np.concatenate(all_true), np.concatenate(all_pred), np.concatenate(all_prob)
    return avg_loss, _compute_eval_metrics(y_true, y_pred, y_prob, y_prob.shape[1])

In [ ]:
# --- Merge and Deduplication ---

def merge_data() -> None:
    if OUT_H5.exists() and OUT_CSV.exists():
        print("Dataset already merged.")
        return

    _require_file(TRAIN_CSV); _require_file(TEST_CSV)
    _require_file(TRAIN_H5); _require_file(TEST_H5)

    df_train = pd.read_csv(TRAIN_CSV)
    df_test = pd.read_csv(TEST_CSV)
    df_train["original_split"], df_test["original_split"] = "train", "test"

    full_df = pd.concat([df_train, df_test], ignore_index=True)
    full_df.to_csv(OUT_CSV, index=False)

    with h5py.File(str(OUT_H5), "w") as f_out:
        with h5py.File(str(TRAIN_H5), "r") as f_tr, h5py.File(str(TEST_H5), "r") as f_te:
            tr_imgs, te_imgs = f_tr["images"], f_te["images"]
            dset = f_out.create_dataset("images", shape=(len(tr_imgs)+len(te_imgs), *tr_imgs.shape[1:]), dtype="f")
            dset[:len(tr_imgs)] = tr_imgs[:]
            dset[len(tr_imgs):] = te_imgs[:]
    print(f"Merged data saved to: {OUT_H5}")


def get_h5_hashes(h5_path: Path, total_images: int, chunk_size: int = 5000) -> List[str]:
    hashes = [""] * total_images
    with h5py.File(str(h5_path), "r") as f:
        imgs = f["images"]
        for start in range(0, total_images, chunk_size):
            end = min(start + chunk_size, total_images)
            chunk = imgs[start:end]
            for i, img in enumerate(chunk):
                hashes[start + i] = hashlib.md5(img.tobytes()).hexdigest()
    return hashes

def analyze_duplicates() -> List[str]:
    df = pd.read_csv(OUT_CSV)
    with h5py.File(str(OUT_H5), "r") as f:
        total = int(f["images"].shape[0])

    all_hashes = get_h5_hashes(OUT_H5, total)
    hash_map = defaultdict(list)
    for idx, h in enumerate(all_hashes): hash_map[h].append(idx)

    dups = {h: idxs for h, idxs in hash_map.items() if len(idxs) > 1}
    if dups:
        dup_indices = [i for idxs in dups.values() for i in idxs]
        report_df = df.iloc[dup_indices].copy()
        report_df["md5"] = [all_hashes[i] for i in dup_indices]
        report_df.to_csv(PROCESSED / "duplicates_report.csv", index=False)

        # Check for MD5 appearing in both train and test
        leakage = report_df.groupby("md5")["original_split"].nunique()
        print("[WARNING] Leakage detected!" if (leakage > 1).any() else "[SAFE] No split leakage.")
    return all_hashes

In [ ]:
# --- Split Generation ---
# Deduplicate by original split, then build train/val/test CSVs.

def create_clean_split(all_hashes: List[str]) -> None:
    df = pd.read_csv(OUT_CSV).copy()
    df["abs_ptr"] = range(len(df))  # Unique identifier for each row
    df["md5"] = all_hashes
    df["indication_type"] = df["indication_type"].astype(str).str.strip()

    tr_df_raw = df[df["original_split"] == "train"].copy()
    te_df_raw = df[df["original_split"] == "test"].copy()

    # Deduplicate within each portion
    tr_before, te_before = len(tr_df_raw), len(te_df_raw)
    tr_df = tr_df_raw.drop_duplicates(subset="md5", keep="first")
    te_df = te_df_raw.drop_duplicates(subset="md5", keep="first")
    tr_removed, te_removed = tr_before - len(tr_df), te_before - len(te_df)
    total_removed = tr_removed + te_removed

    print(f"Duplicates removed (within split): train={tr_removed}, test={te_removed}, total={total_removed}")

    # Filter to keep only known classes (Safety check)
    tr_df = tr_df[tr_df["indication_type"].isin(FULL_CLASSES)].copy()

    # Shuffle all available data
    reduced_tr_df = tr_df.sample(frac=1, random_state=42).reset_index(drop=True)

    # Prepare data for stratified split
    split_data = reduced_tr_df[["abs_ptr", "indication_type"]].copy()
    
    label_counts = split_data["indication_type"].value_counts()
    use_stratify = (label_counts.min() >= 2) and (label_counts.shape[0] > 1)
    stratify_labels = split_data["indication_type"] if use_stratify else None
    
    if not use_stratify:
        print("[WARN] Stratified split disabled (insufficient per-class samples).")

    # Split Train -> Train/Val
    train_ptr, val_ptr = train_test_split(
        split_data["abs_ptr"],
        test_size=0.1,
        random_state=42,
        stratify=stratify_labels,
    )

    # Create final split dataframes
    df_train = reduced_tr_df[reduced_tr_df["abs_ptr"].isin(train_ptr)].sample(frac=1, random_state=42)
    df_val = reduced_tr_df[reduced_tr_df["abs_ptr"].isin(val_ptr)].sample(frac=1, random_state=42)

    # Save to CSV
    df_train.to_csv(TRAIN_SPLIT_CSV, index=False)
    df_val.to_csv(VAL_SPLIT_CSV, index=False)
    te_df.to_csv(TEST_SPLIT_CSV, index=False)

    print(f"Datasets finalized: Train({len(df_train)}), Val({len(df_val)}), Test({len(te_df)})")

    # Print class distribution report
    print_class_counts(df, "FULL (merged)")
    print_class_counts(tr_df_raw, "ORIG TRAIN (raw)")
    print_class_counts(te_df_raw, "ORIG TEST (raw)")
    print_class_counts(tr_df, "ORIG TRAIN (deduped)")
    print_class_counts(te_df, "ORIG TEST (deduped)")
    print_class_counts(df_train, "TRAIN SPLIT")
    print_class_counts(df_val, "VAL SPLIT")
    print_class_counts(te_df, "TEST SPLIT")


In [ ]:
# --- Label Mapping Utilities ---

# Path to save label map as JSON file
LABEL_MAP_JSON = PROCESSED / "label_map.json"

# Cache for loaded label maps to avoid rebuilding
_LABEL_MAP_CACHE: Dict[str, Dict[str, int]] = {}

# Raise error if CSV contains labels not in the label map
def _validate_labels(observed: List[str], label_map: Dict[str, int]) -> None:
    unknown = sorted(set(observed) - set(label_map.keys()))
    if unknown:
        raise ValueError(f"Unknown labels in CSV: {unknown}")

# Build label-to-index mapping from CSV, ordered by FULL_CLASSES
def build_label_map_from_full_csv(full_csv_path: Path) -> Dict[str, int]:
    df = pd.read_csv(full_csv_path)
    observed = set(df["indication_type"].astype(str).str.strip().unique().tolist())

    unknown = sorted(observed - set(FULL_CLASSES))
    if unknown:
        raise ValueError(f"CSV has labels outside FULL_CLASSES: {unknown}")

    ordered_present = [name for name in FULL_CLASSES if name in observed]
    if not ordered_present:
        raise ValueError("No valid labels found in CSV.")

    return {name: i for i, name in enumerate(ordered_present)}

# Load label map from cache, or build from CSV and optionally save to JSON
def load_or_create_label_map(csv_path: Path, persist_json: bool = True) -> Dict[str, int]:
    cache_key = str(csv_path.resolve())
    if cache_key in _LABEL_MAP_CACHE:
        return dict(_LABEL_MAP_CACHE[cache_key])

    label_map = build_label_map_from_full_csv(csv_path)
    _LABEL_MAP_CACHE[cache_key] = dict(label_map)

    if persist_json:
        LABEL_MAP_JSON.write_text(
            json.dumps(label_map, ensure_ascii=False, indent=2), encoding="utf-8"
        )
    return dict(label_map)

# Check that all labels in a split CSV exist in the label map
def validate_split_labels(csv_path: Path, label_map: Dict[str, int]) -> None:
    df = pd.read_csv(csv_path)
    observed = df["indication_type"].astype(str).str.strip().unique().tolist()
    _validate_labels([_normalize_label(x) for x in observed], label_map)

#  Validate labels in train, val, and optionally test splits
def validate_common_splits(label_map: Dict[str, int], include_test: bool = True) -> None:
    validate_split_labels(TRAIN_SPLIT_CSV, label_map)
    validate_split_labels(VAL_SPLIT_CSV, label_map)
    if include_test:
        validate_split_labels(TEST_SPLIT_CSV, label_map)


In [ ]:
# --- DataLoader Utilities ---

_TEST_LOADER_CACHE: Dict[str, Tuple[Sized, DataLoader, Path]] = {}

# Enable pin_memory for faster data transfer when using CUDA
def _pin_memory_enabled() -> bool:
    return isinstance(device, torch.device) and device.type == "cuda"

# Set random seeds for DataLoader workers for reproducibility
def _seed_worker(worker_id: int) -> None:
    worker_seed = (int(globals().get("GLOBAL_SEED", 42)) + worker_id) % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)

# Create a seeded PyTorch Generator for DataLoader
def _make_generator(seed_offset: int = 0) -> torch.Generator:
    gen = torch.Generator()
    gen.manual_seed(int(globals().get("GLOBAL_SEED", 42)) + seed_offset)
    return gen

# Create a DataLoader with consistent configuration for reproducibility
def _make_loader(dataset: Dataset, batch_size: int, shuffle: bool = False, seed_offset: int = 0) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=int(TRAIN_CFG.get("num_workers", 0)),
        pin_memory=_pin_memory_enabled(),
        worker_init_fn=_seed_worker,
        generator=_make_generator(seed_offset),
    )

# Create a unique string signature for a label map (used for cache keys)
def _label_map_signature(label_map: Dict[str, int]) -> str:
    ordered = sorted(label_map.items(), key=lambda kv: kv[1])
    return "|".join([f"{k}:{v}" for k, v in ordered])

# Create train and val datasets with their corresponding DataLoaders
def make_split_datasets_and_loaders(label_map: Dict[str, int], batch_size: int) -> Tuple[Sized, Sized, DataLoader, DataLoader]:
    train_ds = TextileDataset(TRAIN_SPLIT_CSV, OUT_H5, label_map=label_map)
    val_ds = TextileDataset(VAL_SPLIT_CSV, OUT_H5, label_map=label_map)
    train_loader = _make_loader(train_ds, batch_size, shuffle=True, seed_offset=0)
    val_loader = _make_loader(val_ds, batch_size, shuffle=False, seed_offset=1)
    return train_ds, val_ds, train_loader, val_loader

# Build a test loader filtered to classes present in the given label_map
def make_filtered_test_dataset_and_loader(
    label_map: Dict[str, int],
    batch_size: int,
    split_tag: str,
) -> Tuple[Sized, DataLoader, Path]:
    cache_key = f"{split_tag}::{batch_size}::{_label_map_signature(label_map)}"
    if cache_key in _TEST_LOADER_CACHE:
        return _TEST_LOADER_CACHE[cache_key]

    df_test = pd.read_csv(TEST_SPLIT_CSV).copy()
    df_test["indication_type"] = df_test["indication_type"].astype(str).str.strip()
    allowed = set(label_map.keys())
    filtered = df_test[df_test["indication_type"].isin(allowed)].copy()

    filtered_csv = PROCESSED / f"test_split_filtered_{split_tag}.csv"
    filtered.to_csv(filtered_csv, index=False)

    test_ds = TextileDataset(filtered_csv, OUT_H5, label_map=label_map)
    test_loader = _make_loader(test_ds, batch_size, shuffle=False, seed_offset=2)
    _TEST_LOADER_CACHE[cache_key] = (test_ds, test_loader, filtered_csv)
    return _TEST_LOADER_CACHE[cache_key]

In [ ]:
# --- CNN Model (baseline) Definition ---

# Custom Dataset for reading 64x64 textile images from H5 via pointers
class TextileDataset(Dataset):
    def __init__(self, csv_path, h5_path, label_map, transform=None):
        self.df = pd.read_csv(csv_path)
        self.h5_path = str(h5_path)
        self.label_map = label_map
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __iter__(self):
        for idx in range(len(self)):
            yield self[idx]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with h5py.File(self.h5_path, "r") as f:
            img = f["images"][int(row["abs_ptr"].item())]  # type: ignore[arg-type]

        img_t = torch.from_numpy(img).float()

        # Standardize to (Channels, Height, Width)
        if img_t.ndim == 2:
            img_t = img_t.unsqueeze(0)
        elif img_t.ndim == 3 and img_t.shape[-1] == 1:
            # Convert (H, W, 1) to (1, H, W)
            img_t = img_t.permute(2, 0, 1)

        if img_t.max() > 1.0:
            img_t /= 255.0

        if self.transform:
            img_t = self.transform(img_t)

        label = self.label_map[_normalize_label(row["indication_type"])]
        return img_t, torch.tensor(label, dtype=torch.long)

# Three-layer CNN architecture for textile defect classification
class TextileBaselineCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 8 * 8, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
# --- ResNet Model Definition ---

# ResNet-18 adapted for 64x64 grayscale textile images with pre-trained weights
class TextileResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.model(x)

In [ ]:
# --- EfficientNet Model Definition ---

# EfficientNet-B2 with pre-trained weights, frozen backbone, and custom classifier head
class TextileEfficientNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)

        # We set output channels to 32 to perfectly match the next layer's expectation.
        self.model.features[0] = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(inplace=True)
        )

        # Freeze backbone for transfer learning
        for param in self.model.parameters():
            param.requires_grad = False

        # Modify head
        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)

In [ ]:
# --- ViT Model Definition ---

# Vision Transformer with pre-trained weights, adapted for grayscale textile images
class TextileViT(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

        # Adapt patch embedding for 1-channel grayscale input
        old_proj = self.model.conv_proj
        self.model.conv_proj = nn.Conv2d(
            1, old_proj.out_channels,
            kernel_size=old_proj.kernel_size,
            stride=old_proj.stride,
            padding=old_proj.padding,
            bias=old_proj.bias is not None
        )

        for param in self.model.parameters():
            param.requires_grad = False
        in_features = self.model.heads.head.in_features
        self.model.heads.head = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)

In [ ]:
# --- Common Data Setup ---

# Merge raw data, remove duplicates, and create train/val/test splits
merge_data()
hashes = analyze_duplicates()
create_clean_split(hashes)

# Build and validate the global label mapping
label_map = load_or_create_label_map(OUT_CSV)
validate_common_splits(label_map, include_test=True)
print("\nlabel_map:", label_map)

# Create shared datasets and dataloaders
train_ds, val_ds, train_loader, val_loader = make_split_datasets_and_loaders(label_map, int(TRAIN_CFG["batch"]))


In [ ]:
# --- Train Baseline CNN ---

# Model Setup
model = TextileBaselineCNN(num_classes=len(label_map)).to(device)
if torchinfo:
    print(torchinfo.summary(model, input_size=(int(TRAIN_CFG["batch"]), 1, 64, 64), verbose=0, col_names=["input_size", "output_size", "num_params", "kernel_size"]))
else:
    print("[INFO] torchinfo not installed. Skipping model summary.")

# Training Objects
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=float(OPTIM_CFG["lr"]), foreach=bool(OPTIM_CFG["foreach"]))
early_stop = EarlyStopping(patience=int(TRAIN_CFG["patience"]), verbose=False, mode="max", metric_name="f1")

# Training Loop
print(f"\nStarting training on: {device}")
pbar = tqdm(range(int(TRAIN_CFG["epochs"])), desc="Baseline CNN")
history = []

for epoch in pbar:
    run_step(model, train_loader, criterion, optimizer, device, is_train=True)
    v_loss, v_metrics = run_step(model, val_loader, criterion, optimizer, device, is_train=False)

    v_metrics["epoch"] = epoch + 1
    history.append(v_metrics)
    pbar.set_postfix(v_f1=_metric_to_str(v_metrics["f1"]), v_acc=f"{v_metrics['accuracy']:.2f}", v_loss=f"{v_loss:.4f}")

    early_stop(v_metrics["f1"], model)
    if early_stop.early_stop:
        pbar.write(f"Early stopping at epoch {epoch + 1}")
        model.load_state_dict(early_stop.best_model_state)
        break

# Save and Report
best = max(history, key=lambda x: x["f1"])
ckpt_path = OUTPUT_DIR / "best_textile_baseline.pth"
hist_path = OUTPUT_DIR / "baseline_cnn_history.csv"

torch.save(model.state_dict(), str(ckpt_path))
pd.DataFrame(history).to_csv(str(hist_path), index=False)

print(f"\n{'='*40}\nTRAINING COMPLETE\n"
      f"Best Epoch: {best['epoch']}\n"
      f"F1: {best['f1']:.4f} | AUPRC: {best['auprc']:.4f} | Acc: {best['accuracy']:.2f}%\n"
      f"Model saved to: {ckpt_path}\n"
      f"History saved to: {hist_path}\n{'='*40}")

In [ ]:
# --- Train ResNet ---

# Model Setup
model = TextileResNet(num_classes=len(label_map)).to(device)
if torchinfo:
    print(torchinfo.summary(model, input_size=(int(TRAIN_CFG["batch"]), 1, 64, 64), verbose=0, col_names=["input_size", "output_size", "num_params", "kernel_size"]))
else:
    print("[INFO] torchinfo not installed. Skipping model summary.")

# Training Objects
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=float(OPTIM_CFG["lr"]), foreach=bool(OPTIM_CFG["foreach"]))
early_stop = EarlyStopping(patience=int(TRAIN_CFG["patience"]), verbose=False, mode="max", metric_name="f1")

# Training Loop
print(f"\nStarting ResNet training on: {device}")
pbar = tqdm(range(int(TRAIN_CFG["epochs"])), desc="ResNet")
history = []

for epoch in pbar:
    run_step(model, train_loader, criterion, optimizer, device, is_train=True)
    v_loss, v_metrics = run_step(model, val_loader, criterion, optimizer, device, is_train=False)

    v_metrics["epoch"] = epoch + 1
    history.append(v_metrics)
    pbar.set_postfix(v_f1=_metric_to_str(v_metrics["f1"]), v_acc=f"{v_metrics['accuracy']:.2f}", v_loss=f"{v_loss:.4f}")

    early_stop(v_metrics["f1"], model)
    if early_stop.early_stop:
        pbar.write(f"Early stopping at epoch {epoch + 1}")
        model.load_state_dict(early_stop.best_model_state)
        break

# Save and Report
best = max(history, key=lambda x: x["f1"])
ckpt_path = OUTPUT_DIR / "best_resnet.pth"
hist_path = OUTPUT_DIR / "resnet_history.csv"

torch.save(model.state_dict(), str(ckpt_path))
pd.DataFrame(history).to_csv(str(hist_path), index=False)

print(f"\n{'='*40}\nRESNET TRAINING COMPLETE\n"
      f"Best Epoch: {best['epoch']}\n"
      f"F1: {best['f1']:.4f} | AUPRC: {best['auprc']:.4f} | Acc: {best['accuracy']:.2f}%\n"
      f"Model saved to: {ckpt_path}\n"
      f"History saved to: {hist_path}\n{'='*40}")

In [ ]:
# --- Train EfficientNet ---

# Model Setup
model = TextileEfficientNet(num_classes=len(label_map)).to(device)

# Use try-except to prevent torchinfo from crashing on modified models
if torchinfo:
    try:
        dummy_input = torch.randn(int(TRAIN_CFG["batch"]), 1, 64, 64).to(device)
        print(torchinfo.summary(model, input_data=dummy_input, col_names=["input_size", "output_size", "num_params", "kernel_size"]))
    except Exception as e:
        print(f"[WARNING] torchinfo.summary failed: {e}. Proceeding to training.")
else:
    print("[INFO] torchinfo not installed. Skipping model summary.")

# Training Objects
# We use filter() to optimize only the unfrozen layers (the head)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=float(OPTIM_CFG["lr"]), foreach=bool(OPTIM_CFG["foreach"]))
early_stop = EarlyStopping(patience=int(TRAIN_CFG["patience"]), verbose=False, mode="max", metric_name="f1")

# Training Loop
print(f"\nStarting EfficientNet training on: {device}")
pbar = tqdm(range(int(TRAIN_CFG["epochs"])), desc="EfficientNet")
history = []

for epoch in pbar:
    run_step(model, train_loader, criterion, optimizer, device, is_train=True)
    v_loss, v_metrics = run_step(model, val_loader, criterion, optimizer, device, is_train=False)

    v_metrics["epoch"] = epoch + 1
    history.append(v_metrics)
    pbar.set_postfix(v_f1=_metric_to_str(v_metrics["f1"]), v_acc=f"{v_metrics['accuracy']:.2f}", v_loss=f"{v_loss:.4f}")

    early_stop(v_metrics["f1"], model)
    if early_stop.early_stop:
        pbar.write(f"Early stopping at epoch {epoch + 1}")
        model.load_state_dict(early_stop.best_model_state)
        break

# Save and Report
best = max(history, key=lambda x: x["f1"])
ckpt_path = OUTPUT_DIR / "best_efficientnet.pth"
hist_path = OUTPUT_DIR / "efficientnet_history.csv"

torch.save(model.state_dict(), str(ckpt_path))
pd.DataFrame(history).to_csv(str(hist_path), index=False)

print(f"\n{'='*40}\nEFFICIENTNET TRAINING COMPLETE\n"
      f"Best Epoch: {best['epoch']}\n"
      f"F1: {best['f1']:.4f} | AUPRC: {best['auprc']:.4f} | Acc: {best['accuracy']:.2f}%\n"
      f"Model saved to: {ckpt_path}\n"
      f"History saved to: {hist_path}\n{'='*40}")

In [ ]:
# --- Train ViT ---

# Model Setup
model = TextileViT(num_classes=len(label_map)).to(device)

# Robust torchinfo handling
if torchinfo:
    try:
        dummy_input = torch.randn(int(TRAIN_CFG["batch"]), 1, 64, 64).to(device)
        print(torchinfo.summary(model, input_data=dummy_input, col_names=["input_size", "output_size", "num_params", "kernel_size"]))
    except Exception as e:
        print(f"[WARNING] torchinfo.summary failed: {e}. Proceeding to training.")
else:
    print("[INFO] torchinfo not installed. Skipping model summary.")

# Training Objects
# Use filter() to optimize only the unfrozen head (since backbone is frozen in TextileViT)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=float(OPTIM_CFG["lr"]), foreach=bool(OPTIM_CFG["foreach"]))
early_stop = EarlyStopping(patience=int(TRAIN_CFG["patience"]), verbose=False, mode="max", metric_name="f1")

# Training Loop
print(f"\nStarting ViT training on: {device}")
pbar = tqdm(range(int(TRAIN_CFG["epochs"])), desc="ViT")
history = []

for epoch in pbar:
    run_step(model, train_loader, criterion, optimizer, device, is_train=True)
    v_loss, v_metrics = run_step(model, val_loader, criterion, optimizer, device, is_train=False)

    v_metrics["epoch"] = epoch + 1
    history.append(v_metrics)
    pbar.set_postfix(v_f1=_metric_to_str(v_metrics["f1"]), v_acc=f"{v_metrics['accuracy']:.2f}", v_loss=f"{v_loss:.4f}")

    early_stop(v_metrics["f1"], model)
    if early_stop.early_stop:
        pbar.write(f"Early stopping at epoch {epoch + 1}")
        model.load_state_dict(early_stop.best_model_state)
        break

# Save and Report
best = max(history, key=lambda x: x["f1"])
ckpt_path = OUTPUT_DIR / "best_vit.pth"
hist_path = OUTPUT_DIR / "vit_history.csv"

torch.save(model.state_dict(), str(ckpt_path))
pd.DataFrame(history).to_csv(str(hist_path), index=False)

print(f"\n{'='*40}\nViT TRAINING COMPLETE\n"
      f"Best Epoch: {best['epoch']}\n"
      f"F1: {best['f1']:.4f} | AUPRC: {best['auprc']:.4f} | Acc: {best['accuracy']:.2f}%\n"
      f"Model saved to: {ckpt_path}\n"
      f"History saved to: {hist_path}\n{'='*40}")

In [ ]:
# --- Final Test Evaluation (All Models) ---

# 1. Prepare Test Data
# Create a test loader that matches the label_map used for training.
# make_filtered_test_dataset_and_loader returns (dataset, loader, filtered_csv_path)
try:
    _, test_loader, _ = make_filtered_test_dataset_and_loader(label_map, int(TRAIN_CFG["batch"]), split_tag="final_eval")
except Exception as e:
    print(f"[ERROR] Could not create test loader: {e}")
    test_loader = None

if test_loader is not None:
    # 2. Define Models and File Paths
    # Map display names to (ModelClass, CheckpointFilename, HistoryFilename)
    models_config = {
        "Baseline CNN": (TextileBaselineCNN, "best_textile_baseline.pth", "baseline_cnn_history.csv"),
        "ResNet-18": (TextileResNet, "best_resnet.pth", "resnet_history.csv"),
        "EfficientNet-B2": (TextileEfficientNet, "best_efficientnet.pth", "efficientnet_history.csv"),
        "ViT-B-16": (TextileViT, "best_vit.pth", "vit_history.csv")
    }

    results = []
    criterion = nn.CrossEntropyLoss()

    print(f"\n{'='*60}")
    print(f"Evaluating Models on Test Set")
    print(f"{'='*60}")

    # 3. Loop through models and evaluate
    for model_name, (ModelClass, ckpt_file, hist_file) in models_config.items():
        ckpt_path = OUTPUT_DIR / ckpt_file
        hist_path = OUTPUT_DIR / hist_file

        if not ckpt_path.exists():
            print(f"[SKIP] {model_name}: Checkpoint not found at {ckpt_path}")
            continue

        print(f"Loading {model_name}...")
        # Initialize and load weights
        model = ModelClass(num_classes=len(label_map)).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model.eval()

        # Run evaluation (is_train=False, optimizer=None)
        test_loss, metrics = run_step(model, test_loader, criterion, None, device, is_train=False)

        # Find Best Epoch from history file
        best_epoch = "N/A"
        if hist_path.exists():
            try:
                df_hist = pd.read_csv(hist_path)
                if 'f1' in df_hist.columns and 'epoch' in df_hist.columns:
                    best_idx = df_hist['f1'].idxmax()
                    best_epoch = int(df_hist.loc[best_idx, 'epoch'].item())
            except Exception:
                best_epoch = "Error reading history"

        # Collect results
        results.append({
            "Model": model_name,
            "Best Epoch": best_epoch,
            "Test Loss": f"{test_loss:.4f}",
            "Test Accuracy": f"{metrics['accuracy']:.2f}%",
            "Test F1": f"{metrics['f1']:.4f}",
            "Test AUPRC": f"{metrics['auprc']:.4f}"
        })

    # 4. Generate Summary Table and Save
    if results:
        df_results = pd.DataFrame(results)

        # Display in console
        print(f"\n{'='*60}")
        print("FINAL EVALUATION RESULTS SUMMARY")
        print(f"{'='*60}")
        print(df_results.to_string(index=False))
        print(f"{'='*60}")

        # Save to CSV
        results_csv_path = OUTPUT_DIR / "final_evaluation_results.csv"
        df_results.to_csv(results_csv_path, index=False)
        print(f"\nResults saved to: {results_csv_path}")
    else:
        print("\nNo models were evaluated.")
else:
    print("\nEvaluation failed because test data could not be loaded.")

In [ ]:
# --- Result Visualization and Reports ---

# Helper: collect true labels and predictions from a loader
def collect_predictions(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy().tolist())  # type: ignore[arg-type]
            y_pred.extend(preds.cpu().numpy().tolist())  # type: ignore[arg-type]
    return np.array(y_true), np.array(y_pred)

# Create test loader once
test_ds = TextileDataset(TEST_SPLIT_CSV, OUT_H5, label_map=label_map)
test_loader = _make_loader(test_ds, TRAIN_CFG["batch"], shuffle=False, seed_offset=2)

# Model configurations: (Display Name, Class, Checkpoint Path)
models_config = [
    ("Baseline CNN", TextileBaselineCNN, OUTPUT_DIR / "best_textile_baseline.pth"),
    ("ResNet-18", TextileResNet, OUTPUT_DIR / "best_resnet.pth"),
    ("EfficientNet-B2", TextileEfficientNet, OUTPUT_DIR / "best_efficientnet.pth"),
    ("ViT-B-16", TextileViT, OUTPUT_DIR / "best_vit.pth")
]

# Get class names in correct order
class_names = [k for k, v in sorted(label_map.items(), key=lambda x: x[1])]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

print("Generating Confusion Matrices on Test Set...")

for i, (name, ModelClass, ckpt_path) in enumerate(models_config):
    ax = axes[i]
    if not ckpt_path.exists():
        ax.text(0.5, 0.5, f"Checkpoint missing:\n{ckpt_path.name}",
                ha='center', va='center', color='red', fontsize=12)
        ax.set_title(f"{name} (Skipped)")
        ax.axis('off')
        continue

    print(f"Evaluating {name}...")
    model = ModelClass(num_classes=len(label_map)).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))

    y_true, y_pred = collect_predictions(model, test_loader, device)

    # Compute raw confusion matrix
    cm_raw = confusion_matrix(y_true, y_pred, labels=list(range(len(label_map))))

    # Normalize row-wise: (predicted count) / (true count)
    with np.errstate(divide='ignore', invalid='ignore'):
        cm_pct = cm_raw.astype('float') / cm_raw.sum(axis=1)[:, np.newaxis]
        cm_pct = np.nan_to_num(cm_pct) # Replace NaN (from 0/0) with 0.0

    # Plot heatmap with percentage formatting
    sns.heatmap(cm_pct, annot=True, fmt='.1%', cmap='viridis',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, cbar_kws={'label': 'Recall Rate (Pred / True)'})
    ax.set_title(f'{name} - Test Confusion Matrix', fontsize=14, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)

plt.tight_layout(pad=3.0)
out_img_path = OUTPUT_DIR / "confusion_matrices_4models.png"
plt.savefig(out_img_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n Visualization complete. Image saved to: {out_img_path}")

In [ ]:
# --- Integrated Gradients Utilities ---
# Explain model decisions for defect classes using Integrated Gradients.

# IG configuration
IG_BASELINE_MODE = "zeros"
IG_VIZ_METHOD = "blended_heat_map"
IG_VIZ_SIGN = "all"
NON_DEFECT_CLASS_NAMES = {
    str(FULL_CLASSES[0]).strip().lower()
} if "FULL_CLASSES" in globals() and len(FULL_CLASSES) > 0 else {"good"}


# Convert image arrays to HWC format for visualization
def _to_hwc(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 4:
        if arr.shape[0] != 1:
            raise ValueError(f"Expected batch size 1, got shape={arr.shape}")
        arr = arr[0]

    if arr.ndim == 3 and arr.shape[0] <= 4 and arr.shape[1] > 4 and arr.shape[2] > 4:
        arr = np.transpose(arr, (1, 2, 0))
    elif arr.ndim == 2:
        arr = arr[..., np.newaxis]
    elif arr.ndim != 3:
        raise ValueError(f"Unsupported array shape: {arr.shape}")

    return arr


# Create baseline tensor for Integrated Gradients
def _ig_baseline(input_tensor: torch.Tensor) -> torch.Tensor:
    if IG_BASELINE_MODE == "zeros":
        return torch.zeros_like(input_tensor).to(device)
    raise ValueError(f"Unsupported IG_BASELINE_MODE: {IG_BASELINE_MODE}")


# Resolve target class for attribution
def _resolve_target_class(model: nn.Module, input_tensor: torch.Tensor, target_class: Optional[int] = None) -> int:
    if target_class is not None:
        return int(target_class)
    with torch.no_grad():
        output = model(input_tensor)
    return int(torch.argmax(output, dim=1).item())


# Compute IG attribution for one input image tensor
# Returns: (attributions, img_plot, target_id) in HWC numpy format
def apply_integrated_gradients(
    model: nn.Module,
    input_tensor: torch.Tensor,
    target_class: Optional[int] = None
) -> Tuple[np.ndarray, np.ndarray, int]:
    model.eval()
    ig_input = input_tensor.detach().clone().to(device).requires_grad_(True)
    ig = IntegratedGradients(model)
    baseline = _ig_baseline(ig_input)
    target_id = _resolve_target_class(model, ig_input, target_class)

    attributions, _ = ig.attribute(
        inputs=ig_input,
        baselines=baseline,
        target=target_id,
        return_convergence_delta=True,
    )

    attributions = _to_hwc(attributions.detach().cpu().numpy())
    img_plot = _to_hwc(ig_input.detach().cpu().numpy())
    return attributions, img_plot, target_id


# Prepare image for display with appropriate colormap
def _displayable_img(img_plot: np.ndarray) -> Tuple[np.ndarray, Optional[str]]:
    if img_plot.ndim == 3 and img_plot.shape[-1] == 1:
        return img_plot[..., 0], "gray"
    if img_plot.ndim == 2:
        return img_plot, "gray"
    return img_plot, None


# Visualize IG attribution results with side-by-side heatmap and original image
def plot_ig_results(
    attributions: np.ndarray,
    img_plot: np.ndarray,
    predicted_class: int,
    actual_class: int,
    label_map: Dict[str, int],
    sample_idx: Optional[str] = None,
    save_path: Optional[Path] = None,
    model_name: str = "ViT"
) -> None:
    idx_to_class = {v: k for k, v in label_map.items()}
    pred_name = idx_to_class.get(int(predicted_class), str(predicted_class))
    actual_name = idx_to_class.get(int(actual_class), str(actual_class))
    sample_text = f"Sample {sample_idx}" if sample_idx is not None else "Sample"

    fig = plt.figure(figsize=(10, 5.6))
    fig.patch.set_facecolor("#ffffff")
    gs = fig.add_gridspec(2, 2, height_ratios=[20, 1], width_ratios=[1, 1], hspace=0.08, wspace=0.25)
    ax_heat = fig.add_subplot(gs[0, 0])
    ax_orig = fig.add_subplot(gs[0, 1])
    cax = fig.add_subplot(gs[1, :])

    viz.visualize_image_attr(
        attributions,
        img_plot,
        method=IG_VIZ_METHOD,
        sign=IG_VIZ_SIGN,
        show_colorbar=False,
        title="IG Heatmap",
        plt_fig_axis=(fig, ax_heat),
        use_pyplot=False,
    )
    ax_heat.set_title("IG Heatmap", color="#111111", fontsize=11)
    ax_heat.set_aspect("equal")
    if hasattr(ax_heat, "set_box_aspect"):
        ax_heat.set_box_aspect(1)

    disp_img, cmap = _displayable_img(img_plot)
    ax_orig.imshow(disp_img, cmap=cmap, interpolation="nearest")
    ax_orig.set_title("Original Image", color="#111111", fontsize=11)
    ax_orig.axis("off")
    ax_orig.set_aspect("equal")
    if hasattr(ax_orig, "set_box_aspect"):
        ax_orig.set_box_aspect(1)
    ax_orig.set_xlim(ax_heat.get_xlim())
    ax_orig.set_ylim(ax_heat.get_ylim())

    vmax = float(np.max(np.abs(attributions))) if attributions.size else 1.0
    if vmax == 0:
        vmax = 1.0
    ig_cmap = plt.get_cmap("RdYlGn")
    ig_norm = mcolors.Normalize(vmin=-vmax, vmax=vmax)
    sm = plt.cm.ScalarMappable(cmap=ig_cmap, norm=ig_norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.ax.tick_params(colors="#111111", labelsize=8)
    cbar.set_label("Red: negative | Green: positive", color="#111111", fontsize=9, labelpad=4)

    fig.suptitle(
        f"IG Attribution - {model_name} | Predicted: {pred_name} | Actual: {actual_name}",
        fontsize=12,
        color="#111111",
        y=0.98,
    )

    fig.subplots_adjust(top=0.86, bottom=0.18)
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(save_path), dpi=220, format="png", facecolor="#ffffff", edgecolor="#ffffff", bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()
    plt.close(fig)


# Extract defect class names (excluding non-defect classes like 'good')
def _defect_classes(label_map: Dict[str, int]) -> List[Tuple[str, int]]:
    classes = [
        (str(name), int(idx))
        for name, idx in label_map.items()
        if str(name).strip().lower() not in NON_DEFECT_CLASS_NAMES
    ]
    return classes if classes else [(str(n), int(i)) for n, i in label_map.items()]


# Create a safe filename token from text
def _safe_token(text: str) -> str:
    token = "".join(ch if (ch.isalnum() or ch in "-_") else "_" for ch in str(text))
    while "__" in token:
        token = token.replace("__", "_")
    return token.strip("_") or "na"


# Find first sample index for each target class efficiently
def _first_sample_index_per_class(dataset: Dataset, target_class_ids: set) -> Dict[int, int]:
    sample_indices = {}

    if hasattr(dataset, "df") and hasattr(dataset, "label_map"):
        labels = dataset.df["indication_type"].astype(str).str.strip().tolist()
        for idx, class_name in enumerate(labels):
            label_id = int(dataset.label_map[class_name])
            if label_id in target_class_ids and label_id not in sample_indices:
                sample_indices[label_id] = idx
                if len(sample_indices) == len(target_class_ids):
                    break
        return sample_indices

    for idx in range(len(dataset)):
        _, label = dataset[idx]
        label_id = int(label.item()) if torch.is_tensor(label) else int(label)
        if label_id in target_class_ids and label_id not in sample_indices:
            sample_indices[label_id] = idx
            if len(sample_indices) == len(target_class_ids):
                break
    return sample_indices


# Generate IG visualizations for one sample per defect class
def run_ig_demo_per_defect_class(
    model: nn.Module,
    dataset: Dataset,
    label_map: Dict[str, int],
    device: torch.device,
    output_dir: Path = OUTPUT_DIR,
    model_name: str = "ViT"
) -> None:
    print(f"\n{'='*60}")
    print(f"Running IG Attribution - {model_name}")
    print(f"{'='*60}")

    defect_classes = _defect_classes(label_map)
    target_class_ids = {class_id for _, class_id in defect_classes}
    sample_indices = _first_sample_index_per_class(dataset, target_class_ids)

    if not sample_indices:
        print("[WARN] No class samples found for IG visualization.")
        return

    print(f"Processing {len(sample_indices)} defect classes...\n")

    for class_name, class_id in defect_classes:
        if class_id not in sample_indices:
            continue

        ds_idx = sample_indices[class_id]
        sample_img, true_label = dataset[ds_idx]
        input_tensor = sample_img.unsqueeze(0).to(device)

        attrs, img_plot, pred_cls = apply_integrated_gradients(model, input_tensor)
        true_cls = int(true_label.item()) if torch.is_tensor(true_label) else int(true_label)

        save_file = output_dir / f"ig_{model_name.lower()}_{_safe_token(class_name)}_idx_{ds_idx}.png"
        plot_ig_results(
            attrs, img_plot, pred_cls, true_cls, label_map,
            sample_idx=f"{class_name} (idx={ds_idx})",
            save_path=save_file,
            model_name=model_name,
        )

    print(f"\n{'='*60}")
    print(f"Complete. Results saved to: {output_dir}")
    print(f"{'='*60}")


In [ ]:
# --- IG Run for ViT ---
# Captum import is already at top of notebook; verify availability here
try:
    _ = IntegratedGradients  # Will raise NameError if not imported
    _captum_ready = True
except NameError:
    try:
        from captum.attr import IntegratedGradients
        from captum.attr import visualization as viz
        _captum_ready = True
    except ImportError:
        IntegratedGradients = None
        viz = None
        _captum_ready = False

if not _captum_ready:
    print("[INFO] Captum not available. Install with: pip install captum")
else:
    # Detect ViT model from global namespace
    model_candidates = [
        ("model", globals().get("model")),
        ("vit_model", globals().get("vit_model")),
        ("model_vit", globals().get("model_vit")),
    ]

    selected_model = None
    model_var_name = None

    for var_name, candidate in model_candidates:
        if candidate is not None and isinstance(candidate, nn.Module):
            # Check if it's a ViT model by class name or variable name
            if "ViT" in type(candidate).__name__ or "vit" in var_name.lower():
                selected_model = candidate
                model_var_name = var_name
                break

    # Fallback to any available model
    if selected_model is None and model_candidates[0][1] is not None:
        selected_model = model_candidates[0][1]
        model_var_name = "model"

    train_ds = globals().get("train_ds")
    label_map = globals().get("label_map")

    # Check if all prerequisites are ready
    if selected_model is not None and label_map is not None and train_ds is not None and len(train_ds) > 0:
        print(f"[OK] Using model: '{model_var_name}' ({type(selected_model).__name__})")
        try:
            run_ig_demo_per_defect_class(
                selected_model,
                train_ds,
                label_map,
                device,
                model_name="ViT"
            )
        except Exception as e:
            print(f"[ERROR] IG demo failed: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("[INFO] Prerequisites not ready. Run ViT training cell first.")
